<a href="https://colab.research.google.com/github/sigma-asif/explainable-transport-mode-choice/blob/main/1_model_benchmark_rf_shap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transport Mode — Standalone ML + Random Forest Notebook

This notebook is the **frozen standalone machine-learning analysis** for the final journal model.

### Frozen specification
- 348 respondents.
- 33 conceptual predictors.
- 43 encoded model inputs.
- `Mode_When_Depressed`, `Mode_When_Happy`, `Mode_When_Stressed`, `Mode_When_Urgent`, and `Mode_In_Weather` are retained as **nominal categorical predictors**.
- Each `Mode_When_*` variable uses `Intermediate (code 0)` as the reference category and is represented by three dummy columns for modelling.
- RandomOverSampler is applied **only to training folds**.
- Six-model fixed stratified 5-fold OOF screen is retained.
- Random Forest and LightGBM are compared with repeated stratified 5-fold × 10 CV.
- **Random Forest is frozen as the final classifier.** No new tuning, feature selection, model family, or post-hoc optimisation is performed here.
- Final RF diagnostics use fixed 5-fold OOF predictions and probabilities.
- TreeSHAP is computed only for held-out observations in each fold.
- SHAP is exported at both the 43 encoded-input level and the original 33 conceptual-predictor level.


In [ ]:
# ============================================================
# PYTHON 3.13 SAFE ENVIRONMENT SETUP
# ============================================================
# Why this cell exists:
# Do NOT blindly upgrade NumPy/SciPy inside a live notebook kernel.
# A live upgrade can leave already-loaded binary modules inconsistent.
# This setup pins a conservative Python-3.13-compatible core stack.

import sys
import subprocess
from importlib import metadata

CORE_PINS = {
    "numpy": "2.2.6",
    "scipy": "1.15.3",
    "pandas": "2.2.3",
}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

core_to_install = [
    f"{pkg}=={ver}"
    for pkg, ver in CORE_PINS.items()
    if installed_version(pkg) != ver
]

if core_to_install:
    print("Installing Python-3.13-compatible core scientific stack:")
    for item in core_to_install:
        print("  ", item)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-cache-dir", "--upgrade", *core_to_install
    ])

    print("\nCore packages were changed.")
    print("IMPORTANT: restart the notebook kernel/runtime ONCE, then Run All again.")
    print("Do not continue importing NumPy/SciPy in this same live kernel.")

    # Colab can safely force a runtime restart. Other Jupyter frontends should
    # use Kernel -> Restart and then Run All.
    if "google.colab" in sys.modules:
        import os
        os.kill(os.getpid(), 9)
    else:
        raise SystemExit(
            "Core packages installed successfully. Restart kernel, then Run All."
        )

# Install project dependencies WITHOUT upgrading the core scientific stack.
PROJECT_PACKAGES = [
    "scikit-learn",
    "imbalanced-learn",
    "xgboost",
    "lightgbm",
    "shap",
    "matplotlib",
]

missing = [pkg for pkg in PROJECT_PACKAGES if installed_version(pkg) is None]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])

print("Environment ready.")
print("Python :", sys.version.split()[0])
for pkg in [*CORE_PINS.keys(), *PROJECT_PACKAGES]:
    print(f"{pkg:18s}: {installed_version(pkg)}")


Environment ready.
Python : 3.13.15
numpy             : 2.2.6
scipy             : 1.15.3
pandas            : 2.2.3
scikit-learn      : 1.6.1
imbalanced-learn  : 0.14.2
xgboost           : 3.4.1
lightgbm          : 4.6.0
shap              : 0.52.0
matplotlib        : 3.10.0


## Python 3.13 compatibility fix

This notebook avoids the previous unsafe `pip install -U pandas numpy scipy ...` pattern.
If the first setup cell installs or changes NumPy/SciPy/Pandas, **restart the kernel/runtime once and then Run All again**. Google Colab will restart automatically; standard Jupyter/Kaggle-style kernels may require a manual **Kernel → Restart**.

The modelling methodology is unchanged by this environment fix.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)


Mounted at /content/drive


## Run configuration

By default this standalone notebook reads the existing **33-conceptual-predictor cleaned dataset**:

`/content/drive/MyDrive/Research/Transport Mode/Dataset/Cleaned_Data_Final_Preprocessed.csv`

If you instead want to start from the raw survey file, change `INPUT_STAGE = "raw_survey"` and set `INPUT_FILE` to that raw CSV. Unknown category labels deliberately raise an error rather than being silently guessed.


## Frozen analysis implementation

This cell contains the complete preprocessing, 33→43 categorical encoding, six-model OOF screen, repeated RF–LightGBM robustness analysis, frozen final RF, calibration, held-out SHAP, conceptual-SHAP aggregation, Local Top-2 role analysis, publication figures, and reference-run QA.


In [ ]:
"""
Q1 TRANSPORT MODE PIPELINE V2
=============================
Purpose
-------
Upgrade the existing 33-feature transport-mode pipeline so that the five
context-specific mode-preference variables are treated as NOMINAL categories,
not as ordinal 0-3 numeric scores.

Key methodological change
-------------------------
Raw/context codes are preserved in the conceptual 33-predictor dataset:
    0 = Intermediate (Easybike/Rickshaw) [reference]
    1 = Public (Bus)
    2 = Low-cost Personal (Bike/Cycle)
    3 = Private/Hired (CNG/Car)

For model fitting each Mode_When_* variable is converted into 3 binary dummy
columns with code 0 as the reference category. Therefore:
    33 conceptual predictors -> 43 encoded model-input columns.

The pipeline keeps RandomOverSampler strictly inside each training fold.
SVM and MNL are scaled only after training-fold oversampling.
All validation folds remain untouched.

Outputs
-------
1) conceptual_33_features.csv
2) model_matrix_43_features.csv
3) six-model OOF and fold metrics
4) repeated RF-vs-LightGBM 5x10 robustness metrics
5) final RF OOF predictions + calibration
6) encoded 43-column OOF SHAP
7) SHAP aggregated back to the original conceptual 33 predictors
8) respondent-level Local Top-2 dominant-role output using conceptual SHAP

The conceptual SHAP output deliberately aggregates the one-hot Mode_When_*
columns back to their original parent variable. This preserves the scientific
meaning of the original 33 predictors for interpretation and the downstream
LLM reporting layer.
"""

# ---------------------------------------------------------------------------
# OPTIONAL INSTALLATION (Google Colab)
# ---------------------------------------------------------------------------

import os
import re
import json
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import t

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.svm import SVC

from imblearn.over_sampling import RandomOverSampler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

try:
    import shap
except Exception:
    shap = None

warnings.filterwarnings("ignore")


# ===========================================================================
# 1. USER SETTINGS
# ===========================================================================

RANDOM_STATE = 42
K_FOLDS = 5
N_REPEATS = 10
N_ESTIMATORS = 100
N_CALIBRATION_BINS = 10
RUN_REPEATED_CV = True
RUN_SHAP = True

# Default path matches the user's current notebook.
INPUT_FILE = (
    "/content/drive/MyDrive/Research/Transport Mode/"
    "Dataset/Cleaned_Data_Final_Preprocessed.csv"
)

# Use "current_33" for the user's current preprocessed dataset.
# Use "raw_survey" only if the raw survey contains the columns handled below.
INPUT_STAGE = "current_33"  # "current_33" or "raw_survey"

OUTPUT_DIR = (
    "/content/drive/MyDrive/Research/Transport Mode/"
    "FINAL TRANSPORT MODE/Q1 PIPELINE V2"
)

PREPROCESS_DIR = os.path.join(OUTPUT_DIR, "preprocessing")
SIX_MODEL_DIR = os.path.join(OUTPUT_DIR, "six_model_screen")
REPEATED_DIR = os.path.join(OUTPUT_DIR, "repeated_rf_lgbm")
FINAL_RF_DIR = os.path.join(OUTPUT_DIR, "final_rf")
FINAL_RF_METRIC_DIR = os.path.join(FINAL_RF_DIR, "metrics")
DOMINANT_DIR = os.path.join(OUTPUT_DIR, "dominant_role", "metrics")

for directory in [
    PREPROCESS_DIR,
    SIX_MODEL_DIR,
    REPEATED_DIR,
    FINAL_RF_DIR,
    FINAL_RF_METRIC_DIR,
    DOMINANT_DIR,
]:
    os.makedirs(directory, exist_ok=True)


# ===========================================================================
# 2. CANONICAL COLUMN DEFINITIONS
# ===========================================================================

TARGET = "Transport_Grouped"

MODE_CONTEXT_COLS = [
    "Mode_When_Depressed",
    "Mode_When_Happy",
    "Mode_When_Stressed",
    "Mode_When_Urgent",
    "Mode_In_Weather",
]

MODE_CODE_TO_CLASS = {
    0: "Intermediate",
    1: "Public",
    2: "Low-cost Personal",
    3: "Private/Hired",
}

MODE_DUMMY_SUFFIX = {
    1: "Public",
    2: "LowCostPersonal",
    3: "PrivateHired",
}

# Non-mode predictors in the user's current 33-feature dataset.
BASE_19_FEATURES = [
    "Gender",
    "Age",
    "Monthly Income",
    "Average Expendature for transportation",
    "Importance of Cost in Mode Choice",
    "Effect of Cost on Mode Choice",
    "Daily Activity Influence  Mode choice",
    "Distance Influence on Mode Choice",
    "Importance of Safety in Mode Choice",
    "Mobility Barrier on Mode Choice",
    "Accident Memories On Mode Choice",
    "Environmental Impact on Mode Choice",
    "Trust level based on safety in Bus",
    "Trust level based on safety in CNG",
    "Trust level based on safety in Easybike",
    "Trust level based on safety in Rickshaw",
    "Trust level based on safety in Bike",
    "Trust level based on safety in Car",
    "Trust level based on safety in Cycle",
]

OCC_DUMMIES = [
    "Occ_Agriculture",
    "Occ_Business_SelfEmployed",
    "Occ_Homemaker",
    "Occ_Professional",
    "Occ_Service_Labour",
    "Occ_Student",
]

TRAVEL_DUMMIES = [
    "TravelObj_Medical_Recreation",
    "TravelObj_Shopping",
    "TravelObj_Work",
]

CONCEPTUAL_FEATURES_33 = (
    BASE_19_FEATURES
    + MODE_CONTEXT_COLS
    + OCC_DUMMIES
    + TRAVEL_DUMMIES
)

MODE_DUMMY_COLS = [
    f"{parent}__{MODE_DUMMY_SUFFIX[code]}"
    for parent in MODE_CONTEXT_COLS
    for code in [1, 2, 3]
]

MODEL_FEATURES_43 = (
    BASE_19_FEATURES
    + MODE_DUMMY_COLS
    + OCC_DUMMIES
    + TRAVEL_DUMMIES
)

FORBIDDEN_PREDICTORS = [
    "Origin(Ward No)",
    "Destination (ward No)",
    "Income Source",
    "Usually selected specific Transport Mode",
    "Usually Selected Specific Transport Mode",
    "Usually Avoided specific Transport Mode",
    "Usually Avoided Specific Transport Mode",
    "Occupation",
    "Travel Objective",
]

BINARY_FEATURES = [
    "Effect of Cost on Mode Choice",
    "Daily Activity Influence  Mode choice",
    "Distance Influence on Mode Choice",
    "Mobility Barrier on Mode Choice",
    "Accident Memories On Mode Choice",
]

ORDINAL_FEATURES = [
    "Importance of Cost in Mode Choice",
    "Importance of Safety in Mode Choice",
    "Environmental Impact on Mode Choice",
    "Trust level based on safety in Bus",
    "Trust level based on safety in CNG",
    "Trust level based on safety in Easybike",
    "Trust level based on safety in Rickshaw",
    "Trust level based on safety in Bike",
    "Trust level based on safety in Car",
    "Trust level based on safety in Cycle",
]

CONTINUOUS_FEATURES = [
    "Age",
    "Monthly Income",
    "Average Expendature for transportation",
]


# ===========================================================================
# 3. RAW-DATA NORMALIZATION HELPERS
# ===========================================================================

def _norm_text(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s


def _to_numeric_series(series, col_name):
    converted = pd.to_numeric(series, errors="coerce")
    if converted.isna().any():
        bad = series[converted.isna()].dropna().astype(str).unique()[:10]
        raise ValueError(
            f"{col_name}: non-numeric values remain after conversion. Examples: {bad}"
        )
    return converted


def _map_binary(series, col_name):
    mapping = {
        "0": 0,
        "no": 0,
        "n": 0,
        "false": 0,
        "1": 1,
        "yes": 1,
        "y": 1,
        "true": 1,
    }

    def convert(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, (int, np.integer)) and int(v) in [0, 1]:
            return int(v)
        if isinstance(v, (float, np.floating)) and float(v) in [0.0, 1.0]:
            return int(v)
        key = _norm_text(v)
        if key in mapping:
            return mapping[key]
        raise ValueError(f"{col_name}: unknown binary value {v!r}")

    return series.map(convert).astype(int)


def _map_gender(series):
    mapping = {
        "0": 0,
        "female": 0,
        "f": 0,
        "woman": 0,
        "1": 1,
        "male": 1,
        "m": 1,
        "man": 1,
    }

    def convert(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, (int, np.integer)) and int(v) in [0, 1]:
            return int(v)
        if isinstance(v, (float, np.floating)) and float(v) in [0.0, 1.0]:
            return int(v)
        key = _norm_text(v)
        if key in mapping:
            return mapping[key]
        raise ValueError(f"Gender: unknown value {v!r}")

    return series.map(convert).astype(int)


def _map_ordinal_0_4(series, col_name):
    mapping = {
        "0": 0,
        "no effect": 0,
        "none": 0,
        "1": 1,
        "low": 1,
        "2": 2,
        "medium": 2,
        "moderate": 2,
        "3": 3,
        "high": 3,
        "4": 4,
        "extreme": 4,
        "very high": 4,
    }

    def convert(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, (int, np.integer)) and int(v) in [0, 1, 2, 3, 4]:
            return int(v)
        if isinstance(v, (float, np.floating)) and float(v).is_integer() and int(v) in [0, 1, 2, 3, 4]:
            return int(v)
        key = _norm_text(v)
        if key in mapping:
            return mapping[key]
        raise ValueError(f"{col_name}: unknown ordinal value {v!r}")

    return series.map(convert).astype(int)


MODE_ALIASES = {
    # 0: Intermediate
    "0": 0,
    "intermediate": 0,
    "easy bike": 0,
    "easybike": 0,
    "easy bike/rickshaw": 0,
    "easybike/rickshaw": 0,
    "rickshaw": 0,
    # 1: Public
    "1": 1,
    "public": 1,
    "bus": 1,
    "public bus": 1,
    # 2: Low-cost personal
    "2": 2,
    "low cost personal": 2,
    "low-cost personal": 2,
    "bike": 2,
    "motorcycle": 2,
    "motor bike": 2,
    "motorbike": 2,
    "cycle": 2,
    "bicycle": 2,
    "bike/cycle": 2,
    # 3: Private/Hired
    "3": 3,
    "private/hired": 3,
    "private hired": 3,
    "cng": 3,
    "car": 3,
    "cng/car": 3,
}


def _map_mode_to_code(series, col_name):
    def convert(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, (int, np.integer)) and int(v) in [0, 1, 2, 3]:
            return int(v)
        if isinstance(v, (float, np.floating)) and float(v).is_integer() and int(v) in [0, 1, 2, 3]:
            return int(v)
        key = _norm_text(v)
        # normalizer changes hyphen to spaces
        key = key.replace("low cost personal", "low cost personal")
        if key in MODE_ALIASES:
            return MODE_ALIASES[key]
        # explicit normalized variants
        if key in {"easybike rickshaw", "easy bike rickshaw"}:
            return 0
        if key in {"bike cycle", "bicycle motorcycle", "motorcycle bicycle"}:
            return 2
        if key in {"cng car", "car cng"}:
            return 3
        raise ValueError(f"{col_name}: unknown transport-mode value {v!r}")

    return series.map(convert).astype(int)


def _standardize_target(series):
    canonical = {
        "intermediate": "Intermediate",
        "public": "Public",
        "low cost personal": "Low-cost Personal",
        "low-cost personal": "Low-cost Personal",
        "private/hired": "Private/Hired",
        "private hired": "Private/Hired",
    }

    def convert(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, (int, np.integer, float, np.floating)):
            try:
                iv = int(v)
                if iv in MODE_CODE_TO_CLASS and float(v) == iv:
                    return MODE_CODE_TO_CLASS[iv]
            except Exception:
                pass
        key = _norm_text(v)
        if key in canonical:
            return canonical[key]
        # Raw original seven-mode target is also accepted.
        code = _map_mode_to_code(pd.Series([v]), "target").iloc[0]
        return MODE_CODE_TO_CLASS[int(code)]

    return series.map(convert)


# ===========================================================================
# 4. RAW SURVEY PREPROCESSING (OPTIONAL)
# ===========================================================================

OCCUPATION_GROUP_MAP = {
    "advocate": "Professional",
    "doctor": "Professional",
    "teacher": "Professional",
    "business": "Business_SelfEmployed",
    "tailor": "Business_SelfEmployed",
    "driver": "Service_Labour",
    "job": "Service_Labour",
    "labour": "Service_Labour",
    "labor": "Service_Labour",
    "farmer": "Agriculture",
    "homemaker": "Homemaker",
    "housewife": "Homemaker",
    "student": "Student",
    "unemployed": "Unemployed",
}


def _occupation_group(series):
    def convert(v):
        key = _norm_text(v)
        if key in OCCUPATION_GROUP_MAP:
            return OCCUPATION_GROUP_MAP[key]
        raise ValueError(f"Occupation: unknown raw category {v!r}")
    return series.map(convert)


TRAVEL_GROUP_MAP = {
    "education": "Education",
    "study": "Education",
    "work": "Work",
    "job": "Work",
    "shopping": "Shopping",
    "shop": "Shopping",
    "medical": "Medical_Recreation",
    "recreation": "Medical_Recreation",
    "medical/recreation": "Medical_Recreation",
    "medical recreation": "Medical_Recreation",
}


def _travel_group(series):
    def convert(v):
        key = _norm_text(v)
        if key in TRAVEL_GROUP_MAP:
            return TRAVEL_GROUP_MAP[key]
        raise ValueError(f"Travel Objective: unknown raw category {v!r}")
    return series.map(convert)


def preprocess_raw_survey(df_raw):
    """
    Best-effort raw-survey preprocessing based strictly on the categories
    documented in the current methodology. Unknown raw labels intentionally
    raise an error instead of being silently guessed.
    """
    df = df_raw.copy()

    # -----------------------------------------------------------------------
    # Target construction
    # -----------------------------------------------------------------------
    if TARGET in df.columns:
        df[TARGET] = _standardize_target(df[TARGET])
    else:
        target_candidates = [
            "Usually selected specific Transport Mode",
            "Usually Selected Specific Transport Mode",
        ]
        source = next((c for c in target_candidates if c in df.columns), None)
        if source is None:
            raise ValueError(
                f"Raw data must contain {TARGET!r} or one of {target_candidates}."
            )
        df[TARGET] = _standardize_target(df[source])

    # -----------------------------------------------------------------------
    # Basic numeric/binary/ordinal variables
    # -----------------------------------------------------------------------
    if "Gender" in df.columns:
        df["Gender"] = _map_gender(df["Gender"])

    for col in CONTINUOUS_FEATURES:
        if col in df.columns:
            df[col] = _to_numeric_series(df[col], col)

    for col in BINARY_FEATURES:
        if col in df.columns:
            df[col] = _map_binary(df[col], col)

    for col in ORDINAL_FEATURES:
        if col in df.columns:
            df[col] = _map_ordinal_0_4(df[col], col)

    # -----------------------------------------------------------------------
    # Context-specific mode preferences remain conceptual 0-3 codes here.
    # They are one-hot encoded only in the MODEL MATRIX stage.
    # -----------------------------------------------------------------------
    for col in MODE_CONTEXT_COLS:
        if col not in df.columns:
            raise ValueError(f"Missing required contextual-mode variable: {col}")
        df[col] = _map_mode_to_code(df[col], col)

    # -----------------------------------------------------------------------
    # Occupation: 12 raw categories -> 7 conceptual groups, with Unemployed
    # as the reference category -> 6 dummies.
    # -----------------------------------------------------------------------
    if not set(OCC_DUMMIES).issubset(df.columns):
        if "Occupation" not in df.columns:
            raise ValueError(
                "Raw survey preprocessing requires 'Occupation' when the six "
                "Occ_* dummy columns are not already present."
            )
        occ_group = _occupation_group(df["Occupation"])
        for dummy in OCC_DUMMIES:
            group = dummy.replace("Occ_", "")
            df[dummy] = (occ_group == group).astype(int)

    # -----------------------------------------------------------------------
    # Travel objective: Education reference -> 3 dummies.
    # -----------------------------------------------------------------------
    if not set(TRAVEL_DUMMIES).issubset(df.columns):
        if "Travel Objective" not in df.columns:
            raise ValueError(
                "Raw survey preprocessing requires 'Travel Objective' when the "
                "TravelObj_* dummy columns are not already present."
            )
        travel_group = _travel_group(df["Travel Objective"])
        df["TravelObj_Medical_Recreation"] = (
            travel_group == "Medical_Recreation"
        ).astype(int)
        df["TravelObj_Shopping"] = (travel_group == "Shopping").astype(int)
        df["TravelObj_Work"] = (travel_group == "Work").astype(int)

    return make_conceptual_33(df)


# ===========================================================================
# 5. CURRENT 33-FEATURE DATASET VALIDATION
# ===========================================================================

def make_conceptual_33(df_input):
    df = df_input.copy()

    if TARGET not in df.columns:
        raise ValueError(f"{TARGET!r} not found.")

    df[TARGET] = _standardize_target(df[TARGET])

    missing = [c for c in CONCEPTUAL_FEATURES_33 if c not in df.columns]
    if missing:
        raise ValueError(
            "Missing required conceptual predictors:\n- " + "\n- ".join(missing)
        )

    # Enforce documented encodings.
    df["Gender"] = _map_gender(df["Gender"])

    for col in CONTINUOUS_FEATURES:
        df[col] = _to_numeric_series(df[col], col)

    for col in BINARY_FEATURES:
        df[col] = _map_binary(df[col], col)

    for col in ORDINAL_FEATURES:
        df[col] = _map_ordinal_0_4(df[col], col)

    for col in MODE_CONTEXT_COLS:
        df[col] = _map_mode_to_code(df[col], col)

    for col in OCC_DUMMIES + TRAVEL_DUMMIES:
        df[col] = _map_binary(df[col], col)

    # Structural checks on existing one-hot groups.
    if (df[OCC_DUMMIES].sum(axis=1) > 1).any():
        raise ValueError("Invalid occupation dummy structure: >1 active category.")
    if (df[TRAVEL_DUMMIES].sum(axis=1) > 1).any():
        raise ValueError("Invalid travel-objective dummy structure: >1 active category.")

    # Missing values are not silently imputed because the current methodology
    # states the analysis uses 348 complete valid records.
    analysis = df[CONCEPTUAL_FEATURES_33 + [TARGET]].copy()
    if analysis.isna().any().any():
        missing_counts = analysis.isna().sum()
        missing_counts = missing_counts[missing_counts > 0]
        raise ValueError(
            "Missing values found in final analytical variables:\n"
            + missing_counts.to_string()
        )

    # No direct target/preference leakage columns are retained.
    for col in FORBIDDEN_PREDICTORS:
        if col in analysis.columns:
            raise AssertionError(f"Forbidden predictor survived preprocessing: {col}")

    # Validate target classes.
    expected_classes = set(MODE_CODE_TO_CLASS.values())
    actual_classes = set(analysis[TARGET].unique())
    if not actual_classes.issubset(expected_classes):
        raise ValueError(f"Unexpected target classes: {actual_classes - expected_classes}")

    return analysis


# ===========================================================================
# 6. MODE_WHEN_* NOMINAL ONE-HOT ENCODING
# ===========================================================================

def build_model_matrix_43(conceptual_df):
    """
    Convert each Mode_When_* code into three dummy columns using
    Intermediate (code 0) as the reference category.

    No sample-derived category discovery is used: the category set 0/1/2/3 is
    pre-specified from the study design. Therefore this deterministic encoding
    does not use target information or validation-fold statistics.
    """
    model_df = conceptual_df.copy()

    for parent in MODE_CONTEXT_COLS:
        codes = model_df[parent].astype(int)
        for code in [1, 2, 3]:
            new_col = f"{parent}__{MODE_DUMMY_SUFFIX[code]}"
            model_df[new_col] = (codes == code).astype(int)
        model_df.drop(columns=[parent], inplace=True)

    # Exact column order for reproducibility.
    model_df = model_df[MODEL_FEATURES_43 + [TARGET]].copy()

    if len(MODEL_FEATURES_43) != 43:
        raise AssertionError(f"Expected 43 encoded feature names, got {len(MODEL_FEATURES_43)}")

    X = model_df.drop(columns=[TARGET])
    if X.shape[1] != 43:
        raise AssertionError(f"Expected 43 encoded predictors, found {X.shape[1]}")

    for col in MODE_DUMMY_COLS + OCC_DUMMIES + TRAVEL_DUMMIES:
        vals = set(X[col].unique())
        if not vals.issubset({0, 1}):
            raise AssertionError(f"{col} is not binary: {vals}")

    return model_df


# ===========================================================================
# 7. DATA LOADING + PREPROCESSING WRAPPER
# ===========================================================================

def load_and_preprocess(input_file=INPUT_FILE, input_stage=INPUT_STAGE):
    df_input = pd.read_csv(input_file)

    if input_stage == "current_33":
        conceptual = make_conceptual_33(df_input)
    elif input_stage == "raw_survey":
        conceptual = preprocess_raw_survey(df_input)
    else:
        raise ValueError("INPUT_STAGE must be 'current_33' or 'raw_survey'.")

    model_matrix = build_model_matrix_43(conceptual)

    conceptual_path = os.path.join(PREPROCESS_DIR, "conceptual_33_features.csv")
    model_path = os.path.join(PREPROCESS_DIR, "model_matrix_43_features.csv")
    conceptual.to_csv(conceptual_path, index=False)
    model_matrix.to_csv(model_path, index=False)

    print("=" * 88)
    print("PREPROCESSING COMPLETE")
    print("=" * 88)
    print(f"Rows                      : {len(conceptual)}")
    print(f"Conceptual predictors     : {len(CONCEPTUAL_FEATURES_33)}")
    print(f"Encoded model predictors  : {len(MODEL_FEATURES_43)}")
    print("Mode reference category   : Intermediate (code 0)")
    print(f"Saved conceptual dataset  : {conceptual_path}")
    print(f"Saved model matrix        : {model_path}")
    print("\nTarget counts:")
    print(conceptual[TARGET].value_counts())

    print("\nMode_When_* encoded columns:")
    for parent in MODE_CONTEXT_COLS:
        print(f"\n{parent} (reference = Intermediate):")
        for code in [1, 2, 3]:
            print(f"  - {parent}__{MODE_DUMMY_SUFFIX[code]}")

    return conceptual, model_matrix


# ===========================================================================
# 8. COMMON MODEL / METRIC HELPERS
# ===========================================================================

def make_models():
    return {
        "RF": RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE,
        ),
        "MNL": LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            random_state=RANDOM_STATE,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=N_ESTIMATORS,
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            verbosity=0,
            n_jobs=-1,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            verbose=-1,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
        ),
    }


SCALE_SENSITIVE = {"SVM", "MNL"}


def prepare_xy(model_matrix):
    X = model_matrix[MODEL_FEATURES_43].copy()
    label_encoder = LabelEncoder()
    y = pd.Series(
        label_encoder.fit_transform(model_matrix[TARGET]),
        index=model_matrix.index,
        name=TARGET,
    )
    class_names = list(label_encoder.classes_)
    classes = list(range(len(class_names)))
    return X, y, label_encoder, class_names, classes


def fit_fold_model(model_name, estimator, X_train, y_train, X_test):
    """ROS is applied only to the training fold. Validation is untouched."""
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

    scaler = None
    if model_name in SCALE_SENSITIVE:
        scaler = StandardScaler()
        X_train_fit = scaler.fit_transform(X_train_ros)
        X_test_fit = scaler.transform(X_test)
    else:
        X_train_fit = X_train_ros
        X_test_fit = X_test

    model = clone(estimator)
    model.fit(X_train_fit, y_train_ros)
    pred = model.predict(X_test_fit)
    prob = model.predict_proba(X_test_fit)

    return model, scaler, pred, prob, X_train_ros, y_train_ros, X_test_fit


def metric_bundle(y_true, y_pred, y_prob, classes):
    y_bin = label_binarize(y_true, classes=classes)
    try:
        macro_auc = roc_auc_score(
            y_bin,
            y_prob,
            average="macro",
            multi_class="ovr",
        )
    except Exception:
        macro_auc = np.nan

    _, _, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=classes,
        average=None,
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "macro_auc": macro_auc,
        "macro_f1": float(np.mean(f1)),
    }


def safe_class_name(name):
    return str(name).replace("/", "_").replace(" ", "_")


def multiclass_brier(y_true, y_prob, classes):
    y_onehot = label_binarize(y_true, classes=classes)
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))


def top_label_ece(y_true, y_pred, y_prob, n_bins=N_CALIBRATION_BINS):
    confidence = np.max(y_prob, axis=1)
    correct = (np.asarray(y_pred) == np.asarray(y_true)).astype(int)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []

    for b in range(n_bins):
        left, right = edges[b], edges[b + 1]
        if b == n_bins - 1:
            mask = (confidence >= left) & (confidence <= right)
        else:
            mask = (confidence >= left) & (confidence < right)

        n = int(mask.sum())
        if n == 0:
            continue

        acc = float(correct[mask].mean())
        conf = float(confidence[mask].mean())
        ece += (n / len(y_true)) * abs(acc - conf)

        rows.append({
            "bin": b + 1,
            "left": left,
            "right": right,
            "n": n,
            "accuracy": acc,
            "mean_confidence": conf,
            "absolute_gap": abs(acc - conf),
        })

    return float(ece), pd.DataFrame(rows)


# ===========================================================================
# 9. SIX-MODEL FIXED 5-FOLD OOF SCREEN
# ===========================================================================

def run_six_model_screen(model_matrix):
    X, y, le, class_names, classes = prepare_xy(model_matrix)

    cv = StratifiedKFold(
        n_splits=K_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    models = make_models()
    fold_rows = []
    overall_rows = []

    oof_store = {
        name: {
            "pred": np.full(len(X), -1, dtype=int),
            "prob": np.zeros((len(X), len(classes)), dtype=float),
        }
        for name in models
    }

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        for model_name, estimator in models.items():
            model, scaler, pred, prob, X_ros, y_ros, _ = fit_fold_model(
                model_name,
                estimator,
                X_train,
                y_train,
                X_test,
            )

            oof_store[model_name]["pred"][test_idx] = pred
            oof_store[model_name]["prob"][test_idx] = prob

            metrics = metric_bundle(y_test, pred, prob, classes)
            fold_rows.append({
                "fold": fold,
                "model": model_name,
                "train_n_original": len(train_idx),
                "train_n_after_ros": len(X_ros),
                "test_n": len(test_idx),
                **metrics,
            })

    for model_name in models:
        pred = oof_store[model_name]["pred"]
        prob = oof_store[model_name]["prob"]
        metrics = metric_bundle(y, pred, prob, classes)
        overall_rows.append({"model": model_name, **metrics})

        pred_df = pd.DataFrame({
            "row_index": np.arange(len(X)),
            "true_class": le.inverse_transform(y.to_numpy()),
            "predicted_class": le.inverse_transform(pred),
        })
        for j, cname in enumerate(class_names):
            pred_df[f"prob_{safe_class_name(cname)}"] = prob[:, j]
        pred_df.to_csv(
            os.path.join(SIX_MODEL_DIR, f"oof_predictions_{model_name}.csv"),
            index=False,
        )

    fold_df = pd.DataFrame(fold_rows)
    overall_df = pd.DataFrame(overall_rows).sort_values(
        ["macro_f1", "mcc", "accuracy", "macro_auc"],
        ascending=False,
    )

    fold_df.to_csv(os.path.join(SIX_MODEL_DIR, "five_fold_metrics.csv"), index=False)
    overall_df.to_csv(os.path.join(SIX_MODEL_DIR, "oof_overall_metrics.csv"), index=False)

    print("\n" + "=" * 88)
    print("SIX-MODEL FIXED 5-FOLD OOF RESULTS")
    print("=" * 88)
    print(overall_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    return fold_df, overall_df


# ===========================================================================
# 10. REPEATED RF vs LIGHTGBM — 5x10
# ===========================================================================

def run_repeated_rf_lgbm(model_matrix):
    X, y, le, class_names, classes = prepare_xy(model_matrix)

    models = {
        "RF": make_models()["RF"],
        "LightGBM": make_models()["LightGBM"],
    }

    rskf = RepeatedStratifiedKFold(
        n_splits=K_FOLDS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )

    repeat_oof = {
        name: {
            rep: {
                "pred": np.full(len(X), -1, dtype=int),
                "prob": np.zeros((len(X), len(classes)), dtype=float),
            }
            for rep in range(N_REPEATS)
        }
        for name in models
    }

    fold_rows = []

    for split_i, (train_idx, test_idx) in enumerate(rskf.split(X, y)):
        repeat = split_i // K_FOLDS
        fold = (split_i % K_FOLDS) + 1

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        for model_name, estimator in models.items():
            model, scaler, pred, prob, X_ros, y_ros, _ = fit_fold_model(
                model_name,
                estimator,
                X_train,
                y_train,
                X_test,
            )

            repeat_oof[model_name][repeat]["pred"][test_idx] = pred
            repeat_oof[model_name][repeat]["prob"][test_idx] = prob

            fold_rows.append({
                "repeat": repeat + 1,
                "fold": fold,
                "model": model_name,
                **metric_bundle(y_test, pred, prob, classes),
            })

    repeat_rows = []
    for model_name in models:
        for repeat in range(N_REPEATS):
            pred = repeat_oof[model_name][repeat]["pred"]
            prob = repeat_oof[model_name][repeat]["prob"]
            if (pred < 0).any():
                raise AssertionError(f"Incomplete OOF prediction set: {model_name}, repeat {repeat+1}")
            repeat_rows.append({
                "repeat": repeat + 1,
                "model": model_name,
                **metric_bundle(y, pred, prob, classes),
            })

    fold_df = pd.DataFrame(fold_rows)
    repeat_df = pd.DataFrame(repeat_rows)

    summary_rows = []
    for model_name, group in repeat_df.groupby("model"):
        row = {"model": model_name}
        for metric in ["accuracy", "kappa", "mcc", "macro_auc", "macro_f1"]:
            vals = group[metric].to_numpy(float)
            mean = float(vals.mean())
            sd = float(vals.std(ddof=1))
            se = sd / np.sqrt(len(vals))
            crit = float(t.ppf(0.975, df=len(vals) - 1))
            row[f"{metric}_mean"] = mean
            row[f"{metric}_sd"] = sd
            row[f"{metric}_ci_low"] = mean - crit * se
            row[f"{metric}_ci_high"] = mean + crit * se
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    # Paired repeat-level RF - LightGBM differences.
    rf = repeat_df[repeat_df.model == "RF"].sort_values("repeat")
    lg = repeat_df[repeat_df.model == "LightGBM"].sort_values("repeat")

    diff_rows = []
    for metric in ["accuracy", "kappa", "mcc", "macro_auc", "macro_f1"]:
        diff = rf[metric].to_numpy() - lg[metric].to_numpy()
        mean = float(diff.mean())
        sd = float(diff.std(ddof=1))
        se = sd / np.sqrt(len(diff))
        crit = float(t.ppf(0.975, df=len(diff) - 1))
        diff_rows.append({
            "metric": metric,
            "comparison": "RF minus LightGBM",
            "mean_difference": mean,
            "sd_difference": sd,
            "95CI_low": mean - crit * se,
            "95CI_high": mean + crit * se,
            "RF_better_repeats": int((diff > 0).sum()),
            "LightGBM_better_repeats": int((diff < 0).sum()),
            "ties": int((diff == 0).sum()),
        })

    diff_df = pd.DataFrame(diff_rows)

    fold_df.to_csv(os.path.join(REPEATED_DIR, "50_fold_metrics.csv"), index=False)
    repeat_df.to_csv(os.path.join(REPEATED_DIR, "repeat_level_oof_metrics.csv"), index=False)
    summary_df.to_csv(os.path.join(REPEATED_DIR, "repeat_summary_with_descriptive_ci.csv"), index=False)
    diff_df.to_csv(os.path.join(REPEATED_DIR, "paired_repeat_differences.csv"), index=False)

    print("\n" + "=" * 88)
    print("REPEATED RF vs LIGHTGBM — 5-FOLD x 10")
    print("=" * 88)
    display_cols = [
        "model",
        "accuracy_mean", "accuracy_sd",
        "mcc_mean", "mcc_sd",
        "macro_auc_mean", "macro_auc_sd",
        "macro_f1_mean", "macro_f1_sd",
    ]
    print(summary_df[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\nPaired repeat-level differences:")
    print(diff_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    return fold_df, repeat_df, summary_df, diff_df


# ===========================================================================
# 11. SHAP SHAPE NORMALIZATION
# ===========================================================================

def normalize_multiclass_shap(shap_values, n_samples, n_features, n_classes):
    """Return SHAP array with shape (n_samples, n_features, n_classes)."""
    if isinstance(shap_values, list):
        # Legacy SHAP: one [n_samples, n_features] array per class.
        arr = np.stack([np.asarray(v) for v in shap_values], axis=2)
        return arr

    arr = np.asarray(shap_values)

    if arr.shape == (n_samples, n_features, n_classes):
        return arr

    if arr.shape == (n_classes, n_samples, n_features):
        return np.transpose(arr, (1, 2, 0))

    if n_classes == 2 and arr.shape == (n_samples, n_features):
        # This case is not expected for the present four-class problem.
        # Retained only as a defensive branch.
        return np.stack([-arr, arr], axis=2)

    raise ValueError(
        "Unexpected SHAP shape: "
        f"{arr.shape}; expected ({n_samples}, {n_features}, {n_classes})"
    )


# ===========================================================================
# 12. ENCODED -> CONCEPTUAL SHAP AGGREGATION
# ===========================================================================

def encoded_parent(feature):
    for parent in MODE_CONTEXT_COLS:
        if feature.startswith(parent + "__"):
            return parent
    return feature


ENCODED_TO_CONCEPTUAL = {f: encoded_parent(f) for f in MODEL_FEATURES_43}


def aggregate_shap_to_conceptual(encoded_shap):
    """
    Sum SIGNED SHAP values across dummy columns belonging to the same original
    Mode_When_* variable, then return the original 33 conceptual predictors.
    """
    n, p, c = encoded_shap.shape
    if p != len(MODEL_FEATURES_43):
        raise ValueError("Encoded SHAP feature dimension does not match 43 model columns.")

    out = np.zeros((n, len(CONCEPTUAL_FEATURES_33), c), dtype=float)
    conceptual_index = {f: j for j, f in enumerate(CONCEPTUAL_FEATURES_33)}

    for encoded_j, encoded_name in enumerate(MODEL_FEATURES_43):
        parent = ENCODED_TO_CONCEPTUAL[encoded_name]
        parent_j = conceptual_index[parent]
        out[:, parent_j, :] += encoded_shap[:, encoded_j, :]

    return out


# ===========================================================================
# 13. FINAL RF FIXED OOF + CALIBRATION + SHAP
# ===========================================================================

def run_final_rf(conceptual_df, model_matrix):
    X, y, le, class_names, classes = prepare_xy(model_matrix)

    cv = StratifiedKFold(
        n_splits=K_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rf_template = make_models()["RF"]

    oof_pred = np.full(len(X), -1, dtype=int)
    oof_prob = np.zeros((len(X), len(classes)), dtype=float)
    fold_rows = []

    encoded_shap = None
    if RUN_SHAP:
        if shap is None:
            raise ImportError("RUN_SHAP=True but the 'shap' package is unavailable.")
        encoded_shap = np.zeros(
            (len(X), len(MODEL_FEATURES_43), len(classes)),
            dtype=float,
        )

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        rf, scaler, pred, prob, X_ros, y_ros, X_test_fit = fit_fold_model(
            "RF",
            rf_template,
            X_train,
            y_train,
            X_test,
        )

        oof_pred[test_idx] = pred
        oof_prob[test_idx] = prob

        fold_rows.append({
            "fold": fold,
            "train_n_original": len(train_idx),
            "train_n_after_ros": len(X_ros),
            "test_n": len(test_idx),
            **metric_bundle(y_test, pred, prob, classes),
        })

        if RUN_SHAP:
            explainer = shap.TreeExplainer(rf)
            raw_sv = explainer.shap_values(X_test_fit)
            sv = normalize_multiclass_shap(
                raw_sv,
                n_samples=len(test_idx),
                n_features=len(MODEL_FEATURES_43),
                n_classes=len(classes),
            )
            encoded_shap[test_idx, :, :] = sv

    if (oof_pred < 0).any():
        raise AssertionError("Final RF OOF prediction array is incomplete.")

    overall = metric_bundle(y, oof_pred, oof_prob, classes)
    brier = multiclass_brier(y, oof_prob, classes)
    ece, calibration_bins = top_label_ece(y, oof_pred, oof_prob)

    y_true_names = le.inverse_transform(y.to_numpy())
    pred_names = le.inverse_transform(oof_pred)
    confidence = np.max(oof_prob, axis=1)
    correct = (oof_pred == y.to_numpy()).astype(int)

    prediction_df = pd.DataFrame({
        "row_index": np.arange(len(X)),
        "true_class": y_true_names,
        "predicted_class": pred_names,
        "predicted_confidence": confidence,
        "correct": correct,
    })
    for j, cname in enumerate(class_names):
        prediction_df[f"prob_{safe_class_name(cname)}"] = oof_prob[:, j]

    prediction_df.to_csv(
        os.path.join(FINAL_RF_METRIC_DIR, "oof_respondent_predictions.csv"),
        index=False,
    )

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(os.path.join(FINAL_RF_METRIC_DIR, "fold_metrics.csv"), index=False)
    calibration_bins.to_csv(
        os.path.join(FINAL_RF_METRIC_DIR, "calibration_bins.csv"),
        index=False,
    )

    overall_df = pd.DataFrame([{
        **overall,
        "brier_score": brier,
        "ece": ece,
        "mean_confidence_correct": float(confidence[correct == 1].mean()),
        "mean_confidence_incorrect": float(confidence[correct == 0].mean())
        if (correct == 0).any() else np.nan,
    }])
    overall_df.to_csv(
        os.path.join(FINAL_RF_METRIC_DIR, "overall_oof_metrics.csv"),
        index=False,
    )

    # Per-class metrics.
    precision, recall, f1, support = precision_recall_fscore_support(
        y,
        oof_pred,
        labels=classes,
        zero_division=0,
    )
    class_df = pd.DataFrame({
        "class": class_names,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support,
    })
    class_df.to_csv(
        os.path.join(FINAL_RF_METRIC_DIR, "per_class_metrics.csv"),
        index=False,
    )

    cm = confusion_matrix(y, oof_pred, labels=classes)
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
        os.path.join(FINAL_RF_METRIC_DIR, "confusion_matrix.csv")
    )

    print("\n" + "=" * 88)
    print("FINAL RF — FIXED 5-FOLD OOF")
    print("=" * 88)
    print(overall_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\nPer-class metrics:")
    print(class_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    conceptual_shap = None

    if RUN_SHAP:
        # -------------------------------------------------------------------
        # A. Encoded 43-column predicted-class SHAP output
        # -------------------------------------------------------------------
        pred_class_shap_43 = np.zeros((len(X), len(MODEL_FEATURES_43)), dtype=float)
        for i in range(len(X)):
            pred_class_shap_43[i, :] = encoded_shap[i, :, oof_pred[i]]

        local43 = pd.DataFrame(
            pred_class_shap_43,
            columns=[f"SHAP__{f}" for f in MODEL_FEATURES_43],
        )
        local43.insert(0, "predicted_class", pred_names)
        local43.insert(0, "true_class", y_true_names)
        local43.insert(0, "row_index", np.arange(len(X)))
        local43.to_csv(
            os.path.join(FINAL_RF_METRIC_DIR, "oof_local_shap_predicted_class_encoded43.csv"),
            index=False,
        )

        # -------------------------------------------------------------------
        # B. Aggregate dummy SHAPs back to original 33 conceptual predictors.
        # -------------------------------------------------------------------
        conceptual_shap = aggregate_shap_to_conceptual(encoded_shap)

        pred_class_shap_33 = np.zeros((len(X), len(CONCEPTUAL_FEATURES_33)), dtype=float)
        for i in range(len(X)):
            pred_class_shap_33[i, :] = conceptual_shap[i, :, oof_pred[i]]

        local33 = pd.DataFrame(
            pred_class_shap_33,
            columns=[f"SHAP__{f}" for f in CONCEPTUAL_FEATURES_33],
        )
        local33.insert(0, "predicted_class", pred_names)
        local33.insert(0, "true_class", y_true_names)
        local33.insert(0, "row_index", np.arange(len(X)))

        # IMPORTANT: this filename intentionally matches the existing Qwen
        # evidence-builder expectation, while its content now reflects the
        # corrected 43-column model aggregated to 33 conceptual predictors.
        local33.to_csv(
            os.path.join(FINAL_RF_METRIC_DIR, "oof_local_shap_predicted_class.csv"),
            index=False,
        )

        # Global encoded importance: mean |SHAP| across samples/classes.
        global43 = pd.DataFrame({
            "feature": MODEL_FEATURES_43,
            "mean_abs_shap": np.mean(np.abs(encoded_shap), axis=(0, 2)),
        }).sort_values("mean_abs_shap", ascending=False)
        global43.to_csv(
            os.path.join(FINAL_RF_METRIC_DIR, "global_shap_importance_encoded43.csv"),
            index=False,
        )

        # Global conceptual importance: aggregate signed dummy SHAP first,
        # THEN take absolute magnitude.
        global33 = pd.DataFrame({
            "feature": CONCEPTUAL_FEATURES_33,
            "mean_abs_shap": np.mean(np.abs(conceptual_shap), axis=(0, 2)),
        }).sort_values("mean_abs_shap", ascending=False)
        global33.to_csv(
            os.path.join(FINAL_RF_METRIC_DIR, "global_shap_importance_conceptual33.csv"),
            index=False,
        )

        # Class-specific conceptual SHAP importance.
        class_rows = []
        for class_pos, cname in enumerate(class_names):
            vals = np.mean(np.abs(conceptual_shap[:, :, class_pos]), axis=0)
            for feat, val in zip(CONCEPTUAL_FEATURES_33, vals):
                class_rows.append({
                    "class": cname,
                    "feature": feat,
                    "mean_abs_shap": float(val),
                })
        class_shap_df = pd.DataFrame(class_rows)
        class_shap_df.to_csv(
            os.path.join(FINAL_RF_METRIC_DIR, "class_specific_shap_conceptual33.csv"),
            index=False,
        )

        print("\nTop conceptual SHAP features:")
        print(global33.head(15).to_string(index=False, float_format=lambda x: f"{x:.6f}"))

    return {
        "label_encoder": le,
        "class_names": class_names,
        "classes": classes,
        "oof_pred": oof_pred,
        "oof_prob": oof_prob,
        "prediction_df": prediction_df,
        "fold_df": fold_df,
        "overall_df": overall_df,
        "per_class_df": class_df,
        "encoded_shap": encoded_shap,
        "conceptual_shap": conceptual_shap,
    }


# ===========================================================================
# 14. DOMINANT ROLE USING CONCEPTUAL 33-PREDICTOR SHAP
# ===========================================================================

ECONOMIC_FEATURES = {
    "Monthly Income",
    "Average Expendature for transportation",
    "Importance of Cost in Mode Choice",
    "Effect of Cost on Mode Choice",
}

SAFETY_FEATURES = {
    "Importance of Safety in Mode Choice",
    "Mobility Barrier on Mode Choice",
    "Accident Memories On Mode Choice",
    "Trust level based on safety in Bus",
    "Trust level based on safety in CNG",
    "Trust level based on safety in Easybike",
    "Trust level based on safety in Rickshaw",
    "Trust level based on safety in Bike",
    "Trust level based on safety in Car",
    "Trust level based on safety in Cycle",
}

PSYCHOLOGICAL_FEATURES = {
    "Mode_When_Depressed",
    "Mode_When_Happy",
    "Mode_When_Stressed",
}

OTHER_FEATURES = (
    set(CONCEPTUAL_FEATURES_33)
    - ECONOMIC_FEATURES
    - SAFETY_FEATURES
    - PSYCHOLOGICAL_FEATURES
)

ROLE_FEATURES = {
    "Economic": ECONOMIC_FEATURES,
    "Safety": SAFETY_FEATURES,
    "Psychological": PSYCHOLOGICAL_FEATURES,
    "Other": OTHER_FEATURES,
}


def compute_dominant_roles(final_rf_result, top_k=2):
    if final_rf_result["conceptual_shap"] is None:
        print("Dominant-role analysis skipped because SHAP was not run.")
        return None

    conceptual_shap = final_rf_result["conceptual_shap"]
    oof_pred = final_rf_result["oof_pred"]
    prediction_df = final_rf_result["prediction_df"]

    feat_index = {f: j for j, f in enumerate(CONCEPTUAL_FEATURES_33)}

    rows = []

    for i in range(len(prediction_df)):
        local = conceptual_shap[i, :, oof_pred[i]]

        role_scores = {}
        role_top = {}

        for role, features in ROLE_FEATURES.items():
            records = []
            for feature in features:
                j = feat_index[feature]
                sv = float(local[j])
                records.append((feature, sv, abs(sv)))
            records.sort(key=lambda x: x[2], reverse=True)
            chosen = records[:top_k]
            role_top[role] = chosen
            role_scores[role] = float(sum(r[2] for r in chosen))

        dominant_role = max(role_scores, key=role_scores.get)
        top_records = role_top[dominant_role]

        row = {
            "Respondent_ID": i,
            "True_Class": prediction_df.loc[i, "true_class"],
            "Predicted_Class": prediction_df.loc[i, "predicted_class"],
            "Dominant_Role_LocalTop2": dominant_role,
            "Dominant_Role_Top2_Absolute_SHAP_Sum": role_scores[dominant_role],
        }

        for rank, (feature, signed, absolute) in enumerate(top_records, start=1):
            row[f"Top{rank}_Feature"] = feature
            row[f"Top{rank}_Signed_SHAP"] = signed
            row[f"Top{rank}_Abs_SHAP"] = absolute

        for role, score in role_scores.items():
            row[f"RoleScore_{role}"] = score

        rows.append(row)

    out = pd.DataFrame(rows)
    path = os.path.join(DOMINANT_DIR, "dominant_role_respondent_level_ros_rf.csv")
    out.to_csv(path, index=False)

    # Top-k sensitivity from the SAME fixed conceptual OOF SHAP.
    sensitivity = []
    for k in [1, 2, 3]:
        counts = defaultdict(int)
        labels = []
        for i in range(len(prediction_df)):
            local = conceptual_shap[i, :, oof_pred[i]]
            scores = {}
            for role, features in ROLE_FEATURES.items():
                vals = sorted(
                    [abs(float(local[feat_index[f]])) for f in features],
                    reverse=True,
                )
                scores[role] = float(sum(vals[:k]))
            winner = max(scores, key=scores.get)
            counts[winner] += 1
            labels.append(winner)
        sensitivity.append({"k": k, **{role: counts[role] for role in ROLE_FEATURES}})

    sensitivity_df = pd.DataFrame(sensitivity)
    sensitivity_df.to_csv(
        os.path.join(DOMINANT_DIR, "dominant_role_topk_sensitivity_counts.csv"),
        index=False,
    )

    print("\n" + "=" * 88)
    print("LOCAL TOP-2 DOMINANT ROLE — CONCEPTUAL 33 SHAP")
    print("=" * 88)
    print(out["Dominant_Role_LocalTop2"].value_counts())
    print(f"Saved: {path}")

    return out


# ===========================================================================
# 15. QWEN/LLM COMPATIBILITY EXPORT
# ===========================================================================

def export_qwen_compatible_conceptual_dataset(conceptual_df):
    """
    The user's existing Qwen evidence builder expects a 33-predictor dataset
    with the original Mode_When_* 0-3 codes so values can be decoded into mode
    labels. We therefore export that conceptual file separately from the actual
    43-column model matrix.
    """
    path = os.path.join(PREPROCESS_DIR, "Cleaned_Data_Final_Preprocessed_Q1_Conceptual33.csv")
    conceptual_df.to_csv(path, index=False)
    print(f"Qwen-compatible conceptual dataset: {path}")
    return path


# ===========================================================================
# 16. MAIN
# ===========================================================================



# ===========================================================================
# 16. FINAL JOURNAL-READY FIGURES + METRIC QA
# ===========================================================================


# ===========================================================================
# 17. FINAL PUBLICATION FIGURES + FROZEN-RUN QA
# ===========================================================================

FIGURE_DIR = os.path.join(OUTPUT_DIR, "final_publication_figures")
FINAL_SUMMARY_DIR = os.path.join(OUTPUT_DIR, "final_summary")
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(FINAL_SUMMARY_DIR, exist_ok=True)

# Confirmed final-run values. These are never used for prediction/model fitting.
# They are used only to warn if a rerun no longer reproduces the frozen analysis.
REFERENCE_FINAL_RF = {
    "accuracy": 0.8879,
    "kappa": 0.7720,
    "mcc": 0.7794,
    "macro_auc": 0.9471,
    "macro_f1": 0.8547,
    "brier_score": 0.2111,
    "ece": 0.1225,
}

REFERENCE_REPEATED_RF = {
    "accuracy_mean": 0.8848,
    "accuracy_sd": 0.0130,
    "mcc_mean": 0.7731,
    "mcc_sd": 0.0272,
    "macro_auc_mean": 0.9486,
    "macro_auc_sd": 0.0062,
    "macro_f1_mean": 0.8338,
    "macro_f1_sd": 0.0256,
}

REFERENCE_CLASS_METRICS = {
    "Intermediate":      {"precision": 0.8805, "recall": 0.9693, "f1_score": 0.9228, "support": 228},
    "Low-cost Personal": {"precision": 0.9375, "recall": 0.8108, "f1_score": 0.8696, "support": 37},
    "Private/Hired":     {"precision": 0.9500, "recall": 0.8261, "f1_score": 0.8837, "support": 23},
    "Public":            {"precision": 0.8667, "recall": 0.6500, "f1_score": 0.7429, "support": 60},
}

REFERENCE_DOMINANT_ROLE_COUNTS = {
    "Economic": 217,
    "Psychological": 80,
    "Safety": 47,
    "Other": 4,
}

# ---------------------------------------------------------------------------
# Publication style copied from the established figure language in the
# previous analysis notebook.
# ---------------------------------------------------------------------------
MM_TO_IN = 1 / 25.4
SINGLE_COL = 90 * MM_TO_IN
DOUBLE_COL = 190 * MM_TO_IN

PALETTE = [
    "#4C72B0",
    "#DD8452",
    "#55A868",
    "#C44E52",
    "#8172B2",
    "#937860",
    "#DA8BC3",
    "#8C8C8C",
]

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif"],
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.1,
    "lines.markersize": 3.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "savefig.dpi": 600,
    "figure.dpi": 150,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def save_figure(fig, name, use_tight_layout=True, pad_inches=0.25):
    """Save a clean title-free 600-dpi PNG and vector PDF."""
    for ax in fig.axes:
        ax.set_title("")
        ax.grid(False)

    if use_tight_layout:
        try:
            fig.tight_layout()
        except Exception:
            pass

    png_path = os.path.join(FIGURE_DIR, f"{name}.png")
    pdf_path = os.path.join(FIGURE_DIR, f"{name}.pdf")
    fig.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=pad_inches)
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=pad_inches)
    plt.close(fig)
    print("Saved:", png_path)
    print("Saved:", pdf_path)
    return png_path, pdf_path


def plot_class_distribution(conceptual_df, save_name="01_class_distribution"):
    order = ["Intermediate", "Public", "Low-cost Personal", "Private/Hired"]
    counts = conceptual_df[TARGET].value_counts().reindex(order).fillna(0).astype(int)

    fig, ax = plt.subplots(figsize=(SINGLE_COL, SINGLE_COL * 0.75))
    x = np.arange(len(order))
    bars = ax.bar(
        x, counts.values,
        color=[PALETTE[i] for i in range(len(order))],
        edgecolor="black", linewidth=0.4,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=25, ha="right")
    ax.set_ylabel("Respondents")
    ax.set_xlabel("")
    ax.set_ylim(0, counts.max() * 1.13)
    for bar, value in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, value + counts.max()*0.018,
                str(int(value)), ha="center", va="bottom", fontsize=6.5)
    return save_figure(fig, save_name)


def plot_six_model_macro_f1(six_overall_df, save_name="02_six_model_macro_f1"):
    plot_df = six_overall_df.sort_values("macro_f1", ascending=True).copy()
    fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.18, SINGLE_COL * 0.92))
    bars = ax.barh(
        plot_df["model"], plot_df["macro_f1"],
        color=PALETTE[0], edgecolor="black", linewidth=0.4,
    )
    ax.set_xlabel("Out-of-fold Macro-F1")
    ax.set_ylabel("")
    ax.set_xlim(0.60, 0.90)
    for bar, value in zip(bars, plot_df["macro_f1"]):
        ax.text(value + 0.004, bar.get_y()+bar.get_height()/2,
                f"{value:.3f}", va="center", fontsize=6.3)
    return save_figure(fig, save_name)


def plot_repeated_macro_f1(repeat_df, save_name="03_repeated_rf_lgbm_macro_f1"):
    models = ["RF", "LightGBM"]
    values = [repeat_df.loc[repeat_df.model == m, "macro_f1"].to_numpy() for m in models]
    fig, ax = plt.subplots(figsize=(SINGLE_COL * 0.92, SINGLE_COL * 0.86))
    bp = ax.boxplot(
        values,
        tick_labels=models,
        patch_artist=True,
        widths=0.52,
        showmeans=True,
        meanprops=dict(marker="o", markerfacecolor="white", markeredgecolor="black", markersize=3.2),
        medianprops=dict(color="black", linewidth=1.0),
        whiskerprops=dict(linewidth=0.8),
        capprops=dict(linewidth=0.8),
        boxprops=dict(linewidth=0.7),
    )
    for patch, color in zip(bp["boxes"], [PALETTE[0], PALETTE[1]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    for i, vals in enumerate(values, start=1):
        jitter = np.linspace(-0.055, 0.055, len(vals))
        ax.scatter(np.full(len(vals), i)+jitter, vals, s=8, facecolors="none", edgecolors="black", linewidths=0.45, zorder=3)
    ax.set_ylabel("Repeat-level OOF Macro-F1")
    ax.set_xlabel("")
    ax.set_ylim(min(min(v) for v in values)-0.015, max(max(v) for v in values)+0.015)
    return save_figure(fig, save_name)


def plot_rf_per_fold(final_result, save_name="04_rf_per_fold_performance"):
    df = final_result["fold_df"].sort_values("fold")
    metrics = ["accuracy", "macro_f1", "kappa", "mcc", "macro_auc"]
    labels = ["Accuracy", "Macro-F1", "Kappa", "MCC", "Macro-AUC"]
    fig, ax = plt.subplots(figsize=(DOUBLE_COL * 0.72, SINGLE_COL * 0.88))
    for i, (metric, label) in enumerate(zip(metrics, labels)):
        ax.plot(df["fold"], df[metric], marker="o", color=PALETTE[i], label=label)
    ax.set_xlabel("Fold")
    ax.set_ylabel("Metric value")
    ax.set_xticks(df["fold"].astype(int).tolist())
    ax.set_ylim(0.55, 1.01)
    ax.legend(frameon=False, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.33))
    return save_figure(fig, save_name, use_tight_layout=False, pad_inches=0.32)


def plot_confusion_matrix_final(final_result, save_name="05_confusion_matrix"):
    class_names = list(final_result["class_names"])
    y_true = final_result["label_encoder"].transform(final_result["prediction_df"]["true_class"])
    y_pred = final_result["oof_pred"]
    cm = confusion_matrix(y_true, y_pred, labels=final_result["classes"])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.15, SINGLE_COL * 1.15))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=30, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]*100:.1f}%)",
                    ha="center", va="center", fontsize=6, color=color)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Row-normalized proportion")
    cbar.ax.grid(False)
    return save_figure(fig, save_name)


def plot_roc_curves_final(final_result, save_name="06_roc_curves"):
    y_true = final_result["label_encoder"].transform(final_result["prediction_df"]["true_class"])
    y_prob = final_result["oof_prob"]
    classes = final_result["classes"]
    class_names = final_result["class_names"]
    y_bin = label_binarize(y_true, classes=classes)

    fig, ax = plt.subplots(figsize=(SINGLE_COL, SINGLE_COL))
    fpr_grid = np.linspace(0, 1, 200)
    tprs = []
    for i, cname in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        auc_i = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=PALETTE[i], label=f"{cname} (AUC = {auc_i:.3f})")
        tprs.append(np.interp(fpr_grid, fpr, tpr))
    macro_tpr = np.mean(tprs, axis=0)
    macro_auc_val = auc(fpr_grid, macro_tpr)
    ax.plot(fpr_grid, macro_tpr, color="black", linestyle="--", label=f"Macro-average (AUC = {macro_auc_val:.3f})")
    ax.plot([0,1],[0,1], color="gray", linestyle=":", linewidth=0.8)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_xlim(0,1); ax.set_ylim(0,1.02)
    ax.legend(loc="lower right", frameon=False, fontsize=5.8)
    return save_figure(fig, save_name)


def plot_pr_curves_final(final_result, save_name="07_precision_recall_curves"):
    y_true = final_result["label_encoder"].transform(final_result["prediction_df"]["true_class"])
    y_prob = final_result["oof_prob"]
    classes = final_result["classes"]
    class_names = final_result["class_names"]
    y_bin = label_binarize(y_true, classes=classes)

    fig, ax = plt.subplots(figsize=(SINGLE_COL, SINGLE_COL))
    for i, cname in enumerate(class_names):
        precision, recall, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        ax.plot(recall, precision, color=PALETTE[i], label=f"{cname} (AP = {ap:.3f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0,1); ax.set_ylim(0,1.02)
    ax.legend(loc="lower left", frameon=False, fontsize=5.8)
    return save_figure(fig, save_name)


def plot_per_class_metrics(final_result, save_name="08_per_class_precision_recall_f1"):
    df = final_result["per_class_df"].copy()
    x = np.arange(len(df))
    width = 0.22
    fig, ax = plt.subplots(figsize=(DOUBLE_COL * 0.76, SINGLE_COL * 0.90))
    for offset, metric, label, color in [
        (-width, "precision", "Precision", PALETTE[0]),
        (0, "recall", "Recall", PALETTE[1]),
        (width, "f1_score", "F1-score", PALETTE[2]),
    ]:
        ax.bar(x+offset, df[metric], width=width, label=label, color=color, edgecolor="black", linewidth=0.35)
    ax.set_xticks(x)
    ax.set_xticklabels(df["class"], rotation=22, ha="right")
    ax.set_ylabel("Score")
    ax.set_ylim(0,1.02)
    ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.34))
    return save_figure(fig, save_name, use_tight_layout=False, pad_inches=0.34)


def plot_reliability_final(final_result, save_name="09_reliability_diagram"):
    bins = pd.read_csv(os.path.join(FINAL_RF_METRIC_DIR, "calibration_bins.csv"))
    overall = final_result["overall_df"].iloc[0]
    fig, ax = plt.subplots(figsize=(SINGLE_COL, SINGLE_COL))
    ax.plot([0,1],[0,1], color="gray", linestyle=":", linewidth=0.9, label="Perfect calibration")
    ax.plot(bins["mean_confidence"], bins["accuracy"], marker="o", color=PALETTE[0],
            label=f"RF OOF (ECE = {overall['ece']:.3f}; Brier = {overall['brier_score']:.3f})")
    ax.set_xlabel("Mean predicted confidence")
    ax.set_ylabel("Empirical accuracy")
    ax.set_xlim(0,1); ax.set_ylim(0,1.02)
    ax.legend(loc="upper left", frameon=False, fontsize=5.8)
    return save_figure(fig, save_name)


def plot_global_conceptual_shap(final_result, save_name="10_shap_global_importance_conceptual33"):
    conceptual_shap = final_result["conceptual_shap"]
    imp = np.mean(np.abs(conceptual_shap), axis=(0,2))
    df = pd.DataFrame({"feature": CONCEPTUAL_FEATURES_33, "mean_abs_shap": imp}).sort_values("mean_abs_shap", ascending=True)
    df.sort_values("mean_abs_shap", ascending=False).to_csv(
        os.path.join(FINAL_SUMMARY_DIR, "final_conceptual33_global_shap.csv"), index=False
    )

    fig, ax = plt.subplots(figsize=(DOUBLE_COL * 0.92, SINGLE_COL * 2.35))
    ax.barh(df["feature"], df["mean_abs_shap"], color=PALETTE[0], edgecolor="black", linewidth=0.4)
    ax.set_xlabel("Mean absolute SHAP value (averaged across classes and folds)", fontsize=8, labelpad=9)
    ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=6.3)
    ax.tick_params(axis="x", labelsize=7)
    fig.subplots_adjust(left=0.42, right=0.97, bottom=0.09, top=0.985)
    return save_figure(fig, save_name, use_tight_layout=False, pad_inches=0.40)


def plot_encoded_predicted_class_beeswarm(model_matrix, final_result, save_name="11_shap_beeswarm_top15_encoded"):
    """
    Beeswarm is intentionally shown on the actual 43 encoded model inputs.
    This avoids using the nominal 0-3 Mode_When_* codes as a continuous colour scale.
    """
    encoded_shap = final_result["encoded_shap"]
    oof_pred = final_result["oof_pred"]
    pred_shap = np.zeros((len(model_matrix), len(MODEL_FEATURES_43)), dtype=float)
    for i in range(len(model_matrix)):
        pred_shap[i, :] = encoded_shap[i, :, oof_pred[i]]

    importance = np.mean(np.abs(pred_shap), axis=0)
    top_idx = np.argsort(importance)[::-1][:15]
    top_features = [MODEL_FEATURES_43[i] for i in top_idx]
    shap_values_top = pred_shap[:, top_idx]
    X_top = model_matrix[top_features].copy()

    pd.DataFrame({"feature": top_features, "mean_abs_predicted_class_shap": importance[top_idx]}).to_csv(
        os.path.join(FINAL_SUMMARY_DIR, "final_top15_encoded_predicted_class_shap.csv"), index=False
    )

    plt.figure(figsize=(DOUBLE_COL * 0.92, SINGLE_COL * 1.55))
    shap.summary_plot(
        shap_values_top,
        X_top,
        feature_names=top_features,
        plot_type="dot",
        max_display=15,
        show=False,
        plot_size=None,
    )
    fig = plt.gcf()
    main_ax = fig.axes[0]
    main_ax.set_xlabel("SHAP value (impact on predicted-class model output)", fontsize=8, labelpad=9)
    main_ax.tick_params(axis="x", labelsize=7)
    main_ax.tick_params(axis="y", labelsize=6.5)
    main_ax.grid(False)
    if len(fig.axes) > 1:
        cbar_ax = fig.axes[-1]
        cbar_ax.tick_params(labelsize=6.5)
        cbar_ax.set_ylabel("Feature value", fontsize=7, labelpad=10)
        cbar_ax.grid(False)
    fig.subplots_adjust(left=0.34, right=0.86, bottom=0.11, top=0.985)
    return save_figure(fig, save_name, use_tight_layout=False, pad_inches=0.35)


def native_shap_beeswarm_on_axis(
    ax,
    shap_values,
    feature_data,
    feature_names_local,
    top_n=15,
):
    """Draw a native SHAP beeswarm on a supplied Matplotlib axis."""
    explanation = shap.Explanation(
        values=np.asarray(shap_values, dtype=float),
        data=np.asarray(feature_data, dtype=float),
        feature_names=list(feature_names_local),
    )

    try:
        shap.plots.beeswarm(
            explanation,
            max_display=top_n,
            order=shap.Explanation.abs.mean(0),
            color=shap.plots.colors.red_blue,
            color_bar=False,
            group_remaining_features=False,
            show=False,
            ax=ax,
            plot_size=None,
        )
    except TypeError:
        # Compatibility fallback for older SHAP releases.
        plt.sca(ax)
        shap.summary_plot(
            np.asarray(shap_values, dtype=float),
            feature_data,
            feature_names=list(feature_names_local),
            plot_type="dot",
            max_display=top_n,
            color_bar=False,
            show=False,
            plot_size=None,
        )

    ax.set_title("")
    ax.grid(False)


def plot_per_class_shap_beeswarm_2x2_encoded(
    model_matrix,
    final_result,
    top_n=15,
    save_name="12_shap_beeswarm_per_class_2x2_encoded",
):
    """
    Four class-specific SHAP beeswarms in one 2x2 publication panel.

    IMPORTANT: this panel uses the actual 43 encoded model inputs rather than
    the raw 0-3 Mode_When_* category identifiers. Therefore a Mode_When_*
    category appears as a binary dummy feature and the beeswarm colour scale
    remains interpretable (0 = absent, 1 = present) instead of implying an
    artificial ordinal relationship among transport-mode categories.
    """
    encoded_shap = final_result["encoded_shap"]
    class_names = final_result["class_names"]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(DOUBLE_COL * 1.65, SINGLE_COL * 3.20),
    )
    axes = axes.flatten()
    panel_letters = ["(a)", "(b)", "(c)", "(d)"]

    class_plot_data = []
    global_abs_max = 0.0

    for class_pos, class_name in enumerate(class_names):
        class_shap_full = encoded_shap[:, :, class_pos]
        class_importance = np.mean(np.abs(class_shap_full), axis=0)
        top_indices = np.argsort(class_importance)[::-1][:top_n]
        top_features = [MODEL_FEATURES_43[i] for i in top_indices]
        class_shap_top = class_shap_full[:, top_indices]
        class_X_top = model_matrix.loc[:, top_features].copy()

        # Save class-specific top-15 encoded ranking used in this panel.
        pd.DataFrame({
            "rank": np.arange(1, len(top_indices) + 1),
            "class": class_name,
            "feature": top_features,
            "mean_abs_shap": class_importance[top_indices],
        }).to_csv(
            os.path.join(
                FINAL_SUMMARY_DIR,
                f"class_shap_beeswarm_top15_encoded_{safe_class_name(class_name).lower()}.csv",
            ),
            index=False,
        )

        finite_values = class_shap_top[np.isfinite(class_shap_top)]
        if finite_values.size:
            global_abs_max = max(global_abs_max, float(np.max(np.abs(finite_values))))

        class_plot_data.append({
            "class_name": class_name,
            "top_features": top_features,
            "class_shap_top": class_shap_top,
            "class_X_top": class_X_top,
        })

    if global_abs_max <= 0:
        global_abs_max = 1.0
    x_padding = 0.06 * global_abs_max
    shared_xlim = (-global_abs_max - x_padding, global_abs_max + x_padding)

    for panel_idx, data in enumerate(class_plot_data):
        ax = axes[panel_idx]
        native_shap_beeswarm_on_axis(
            ax=ax,
            shap_values=data["class_shap_top"],
            feature_data=data["class_X_top"],
            feature_names_local=data["top_features"],
            top_n=top_n,
        )
        ax.set_xlim(*shared_xlim)
        ax.set_xlabel(
            f"SHAP value (impact on {data['class_name']})",
            fontsize=6.8,
            labelpad=6,
        )
        ax.tick_params(axis="x", labelsize=6)
        ax.tick_params(axis="y", labelsize=5.4, pad=3)
        ax.grid(False)
        ax.text(
            0.0,
            1.025,
            f"{panel_letters[panel_idx]} {data['class_name']}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=7.2,
            fontweight="bold",
        )

    # Shared SHAP feature-value colour bar, matching the earlier notebook style.
    try:
        shap_cmap = shap.plots.colors.red_blue
    except Exception:
        shap_cmap = mpl.colors.LinearSegmentedColormap.from_list(
            "shap_blue_red_fallback",
            ["#008BFB", "#7B4CE2", "#FF0051"],
        )

    scalar_mappable = mpl.cm.ScalarMappable(
        norm=mpl.colors.Normalize(vmin=0.0, vmax=1.0),
        cmap=shap_cmap,
    )
    scalar_mappable.set_array([])
    cbar = fig.colorbar(
        scalar_mappable,
        ax=axes.tolist(),
        fraction=0.020,
        pad=0.016,
        aspect=35,
    )
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["Low", "High"])
    cbar.ax.tick_params(labelsize=6.5, length=0)
    cbar.set_label("Feature value", fontsize=7, labelpad=8)
    cbar.outline.set_visible(False)
    cbar.ax.grid(False)

    fig.subplots_adjust(
        left=0.245,
        right=0.925,
        bottom=0.055,
        top=0.975,
        wspace=0.84,
        hspace=0.34,
    )
    return save_figure(
        fig,
        save_name,
        use_tight_layout=False,
        pad_inches=0.30,
    )


def plot_class_specific_conceptual_shap(final_result):
    conceptual_shap = final_result["conceptual_shap"]
    out_paths = []
    for class_pos, cname in enumerate(final_result["class_names"]):
        imp = np.mean(np.abs(conceptual_shap[:, :, class_pos]), axis=0)
        df = pd.DataFrame({"feature": CONCEPTUAL_FEATURES_33, "mean_abs_shap": imp}).sort_values("mean_abs_shap", ascending=False).head(15)
        plot_df = df.sort_values("mean_abs_shap", ascending=True)
        safe = safe_class_name(cname).lower()
        df.to_csv(os.path.join(FINAL_SUMMARY_DIR, f"class_shap_top15_{safe}.csv"), index=False)
        fig, ax = plt.subplots(figsize=(DOUBLE_COL * 0.82, SINGLE_COL * 1.35))
        ax.barh(plot_df["feature"], plot_df["mean_abs_shap"], color=PALETTE[class_pos], edgecolor="black", linewidth=0.35)
        ax.set_xlabel(f"Mean absolute SHAP value — {cname}")
        ax.set_ylabel("")
        ax.tick_params(axis="y", labelsize=6.3)
        fig.subplots_adjust(left=0.40, right=0.97, bottom=0.11, top=0.985)
        out_paths.append(save_figure(fig, f"{13+class_pos:02d}_class_shap_{safe}", use_tight_layout=False, pad_inches=0.35))
    return out_paths


def plot_dominant_role_counts(dominant_df, save_name="17_dominant_role_counts"):
    order = ["Economic", "Psychological", "Safety", "Other"]
    counts = dominant_df["Dominant_Role_LocalTop2"].value_counts().reindex(order).fillna(0).astype(int)
    fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.02, SINGLE_COL * 0.78))
    bars = ax.bar(order, counts.values, color=[PALETTE[0], PALETTE[4], PALETTE[2], PALETTE[7]], edgecolor="black", linewidth=0.4)
    ax.set_ylabel("Respondents")
    ax.set_xlabel("")
    ax.set_ylim(0, counts.max()*1.13)
    ax.tick_params(axis="x", rotation=18)
    for bar, value in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, value+counts.max()*0.018, str(int(value)), ha="center", fontsize=6.5)
    return save_figure(fig, save_name)


def plot_topk_sensitivity(save_name="18_dominant_role_topk_sensitivity"):
    path = os.path.join(DOMINANT_DIR, "dominant_role_topk_sensitivity_counts.csv")
    df = pd.read_csv(path)
    fig, ax = plt.subplots(figsize=(DOUBLE_COL * 0.68, SINGLE_COL * 0.85))
    roles = ["Economic", "Psychological", "Safety", "Other"]
    for i, role in enumerate(roles):
        ax.plot(df["k"], df[role], marker="o", color=PALETTE[[0,4,2,7][i]], label=role)
    ax.set_xlabel("Fixed-k contributions per role")
    ax.set_ylabel("Respondents assigned to role")
    ax.set_xticks(df["k"].astype(int).tolist())
    ax.legend(frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.33))
    return save_figure(fig, save_name, use_tight_layout=False, pad_inches=0.32)


def reference_qa(final_result, repeat_summary_df):
    rows = []
    observed_final = final_result["overall_df"].iloc[0].to_dict()
    for metric, expected in REFERENCE_FINAL_RF.items():
        observed = float(observed_final[metric])
        rows.append({
            "section": "Final RF fixed 5-fold OOF",
            "metric": metric,
            "expected_reference": expected,
            "observed_recomputed": observed,
            "absolute_difference": abs(observed-expected),
            "status": "PASS" if abs(observed-expected) <= 0.0020 else "REVIEW",
        })
    rf_row = repeat_summary_df.loc[repeat_summary_df["model"] == "RF"].iloc[0]
    for metric, expected in REFERENCE_REPEATED_RF.items():
        observed = float(rf_row[metric])
        rows.append({
            "section": "Repeated RF 5x10",
            "metric": metric,
            "expected_reference": expected,
            "observed_recomputed": observed,
            "absolute_difference": abs(observed-expected),
            "status": "PASS" if abs(observed-expected) <= 0.0020 else "REVIEW",
        })
    qa = pd.DataFrame(rows)
    qa.to_csv(os.path.join(FINAL_SUMMARY_DIR, "final_reference_metric_QA.csv"), index=False)
    return qa


def write_final_model_card(repeat_summary_df, final_result, dominant_df, qa_df):
    rf = repeat_summary_df.loc[repeat_summary_df.model == "RF"].iloc[0]
    lgb = repeat_summary_df.loc[repeat_summary_df.model == "LightGBM"].iloc[0]
    overall = final_result["overall_df"].iloc[0]
    roles = dominant_df["Dominant_Role_LocalTop2"].value_counts()
    lines = [
        "FINAL FROZEN TRANSPORT-MODE MODEL CARD",
        "="*72,
        "Final model: Random Forest",
        "Conceptual predictors: 33",
        "Encoded model inputs: 43",
        "Mode_When_*: nominal one-hot encoding, Intermediate (0) reference",
        "Class balancing: RandomOverSampler within training folds only",
        "Primary model-selection metric: Macro-F1",
        "Primary detailed evaluation: fixed stratified 5-fold OOF",
        "Robustness: repeated stratified 5-fold x 10 RF vs LightGBM",
        "",
        "FINAL RF FIXED 5-FOLD OOF",
        f"Accuracy   : {overall['accuracy']:.4f}",
        f"Kappa      : {overall['kappa']:.4f}",
        f"MCC        : {overall['mcc']:.4f}",
        f"Macro-AUC  : {overall['macro_auc']:.4f}",
        f"Macro-F1   : {overall['macro_f1']:.4f}",
        f"Brier      : {overall['brier_score']:.4f}",
        f"ECE        : {overall['ece']:.4f}",
        "",
        "REPEATED 5x10",
        f"RF Macro-F1       : {rf['macro_f1_mean']:.4f} +/- {rf['macro_f1_sd']:.4f}",
        f"LightGBM Macro-F1 : {lgb['macro_f1_mean']:.4f} +/- {lgb['macro_f1_sd']:.4f}",
        "",
        "LOCAL TOP-2 ROLE COUNTS",
    ]
    for role in ["Economic", "Psychological", "Safety", "Other"]:
        lines.append(f"{role:14s}: {int(roles.get(role,0))}")
    lines += [
        "",
        f"Reference QA all passed: {bool((qa_df['status']=='PASS').all())}",
        "",
        "Interpretation boundary: SHAP values are model attributions, not causal effects.",
        "The Local Top-2 role is a descriptive respondent-level attribution summary.",
    ]
    path = os.path.join(FINAL_SUMMARY_DIR, "FINAL_FROZEN_RF_MODEL_CARD.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path


def create_final_publication_outputs(conceptual_df, model_matrix, six_overall_df, repeat_df, repeat_summary_df, diff_df, final_result, dominant_df):
    print("\n" + "="*88)
    print("FINAL PUBLICATION FIGURES + SUMMARY")
    print("="*88)

    # Snapshot all headline tables.
    six_overall_df.to_csv(os.path.join(FINAL_SUMMARY_DIR, "final_six_model_oof_metrics.csv"), index=False)
    repeat_summary_df.to_csv(os.path.join(FINAL_SUMMARY_DIR, "final_repeated_rf_lgbm_summary.csv"), index=False)
    diff_df.to_csv(os.path.join(FINAL_SUMMARY_DIR, "final_paired_repeat_differences.csv"), index=False)
    final_result["overall_df"].to_csv(os.path.join(FINAL_SUMMARY_DIR, "final_rf_oof_overall_metrics.csv"), index=False)
    final_result["per_class_df"].to_csv(os.path.join(FINAL_SUMMARY_DIR, "final_rf_per_class_metrics.csv"), index=False)

    # Publication figures.
    plot_class_distribution(conceptual_df)
    plot_six_model_macro_f1(six_overall_df)
    plot_repeated_macro_f1(repeat_df)
    plot_rf_per_fold(final_result)
    plot_confusion_matrix_final(final_result)
    plot_roc_curves_final(final_result)
    plot_pr_curves_final(final_result)
    plot_per_class_metrics(final_result)
    plot_reliability_final(final_result)
    plot_global_conceptual_shap(final_result)
    plot_encoded_predicted_class_beeswarm(model_matrix, final_result)
    plot_per_class_shap_beeswarm_2x2_encoded(model_matrix, final_result, top_n=15)
    plot_class_specific_conceptual_shap(final_result)
    plot_dominant_role_counts(dominant_df)
    plot_topk_sensitivity()

    qa_df = reference_qa(final_result, repeat_summary_df)
    model_card = write_final_model_card(repeat_summary_df, final_result, dominant_df, qa_df)

    manifest = []
    for folder, artifact_type in [(FIGURE_DIR, "figure"), (FINAL_SUMMARY_DIR, "summary")]:
        for name in sorted(os.listdir(folder)):
            manifest.append({"artifact_type": artifact_type, "filename": name, "path": os.path.join(folder, name)})
    pd.DataFrame(manifest).to_csv(os.path.join(FINAL_SUMMARY_DIR, "final_output_manifest.csv"), index=False)

    print("Figures saved to:", FIGURE_DIR)
    print("Summary saved to:", FINAL_SUMMARY_DIR)
    print("Model card:", model_card)
    print("\nReference QA:")
    print(qa_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
    if (qa_df["status"] == "PASS").all():
        print("\nFINAL QA STATUS: PASS")
    else:
        print("\nFINAL QA STATUS: REVIEW — check dataset/package versions before publication.")
    return qa_df


def main():
    print("\nQ1 TRANSPORT MODE PIPELINE V2 — FINAL FROZEN RF")
    print("Mode_When_* = NOMINAL categorical predictors; code 0 is reference.")
    print("No new tuning/model search is performed in this final notebook.\n")

    conceptual_df, model_matrix = load_and_preprocess()
    export_qwen_compatible_conceptual_dataset(conceptual_df)

    # 1) Six-model fixed 5-fold OOF screen.
    six_fold_df, six_overall_df = run_six_model_screen(model_matrix)

    # 2) RF vs LightGBM repeated 5x10 robustness assessment.
    repeated_fold_df = None
    repeat_df = None
    repeat_summary_df = None
    diff_df = None
    if RUN_REPEATED_CV:
        (
            repeated_fold_df,
            repeat_df,
            repeat_summary_df,
            diff_df,
        ) = run_repeated_rf_lgbm(model_matrix)

    # 3) Frozen final RF: fixed 5-fold OOF + calibration + held-out SHAP.
    final_result = run_final_rf(conceptual_df, model_matrix)

    # 4) Descriptive Local Top-2 role analysis on conceptual 33 SHAP.
    dominant_df = compute_dominant_roles(final_result, top_k=2)

    # 5) Final journal-ready outputs and frozen-run QA.
    qa_df = None
    if repeat_summary_df is not None:
        qa_df = create_final_publication_outputs(
            conceptual_df=conceptual_df,
            model_matrix=model_matrix,
            six_overall_df=six_overall_df,
            repeat_df=repeat_df,
            repeat_summary_df=repeat_summary_df,
            diff_df=diff_df,
            final_result=final_result,
            dominant_df=dominant_df,
        )

    print("\n" + "=" * 88)
    print("FINAL FROZEN PIPELINE FINISHED")
    print("=" * 88)
    print(f"Output directory: {OUTPUT_DIR}")
    print("\nFinal model: Random Forest")
    print("Conceptual predictors: 33")
    print("Encoded model predictors: 43")
    print(
        "Interpretation: models are fitted on 43 encoded columns; SHAP is also "
        "exported at the original 33-conceptual-predictor level by summing signed "
        "SHAP across each Mode_When_* dummy group."
    )

    return {
        "conceptual_df": conceptual_df,
        "model_matrix": model_matrix,
        "six_fold_df": six_fold_df,
        "six_overall_df": six_overall_df,
        "repeated_fold_df": repeated_fold_df,
        "repeat_df": repeat_df,
        "repeat_summary_df": repeat_summary_df,
        "diff_df": diff_df,
        "final_result": final_result,
        "dominant_df": dominant_df,
        "qa_df": qa_df,
    }


## Execute the complete final pipeline

This is the only analysis execution cell. It recomputes every metric and regenerates all saved outputs from the configured input dataset.


In [ ]:
FINAL_PIPELINE_RESULTS = main()


Q1 TRANSPORT MODE PIPELINE V2 — FINAL FROZEN RF
Mode_When_* = NOMINAL categorical predictors; code 0 is reference.
No new tuning/model search is performed in this final notebook.

PREPROCESSING COMPLETE
Rows                      : 348
Conceptual predictors     : 33
Encoded model predictors  : 43
Mode reference category   : Intermediate (code 0)
Saved conceptual dataset  : /content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/preprocessing/conceptual_33_features.csv
Saved model matrix        : /content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/preprocessing/model_matrix_43_features.csv

Target counts:
Transport_Grouped
Intermediate         228
Public                60
Low-cost Personal     37
Private/Hired         23
Name: count, dtype: int64

Mode_When_* encoded columns:

Mode_When_Depressed (reference = Intermediate):
  - Mode_When_Depressed__Public
  - Mode_When_Depressed__LowCostPersonal
  - Mode_When_Depressed__PrivateHi

## Saved outputs

The final run writes to:

`/content/drive/MyDrive/Research/Transport Mode/FINAL TRANSPORT MODE/Q1 PIPELINE V2/`

Key folders:
- `preprocessing/` — conceptual 33-feature data and encoded 43-input matrix.
- `six_model_screen/` — fixed 5-fold OOF metrics and predictions for six classifiers.
- `repeated_rf_lgbm/` — 5×10 repeated RF vs LightGBM fold/repeat metrics and paired differences.
- `final_rf/metrics/` — frozen RF OOF predictions, calibration, confusion matrix and SHAP exports.
- `dominant_role/metrics/` — Local Top-2 role results and Top-k sensitivity.
- `final_publication_figures/` — publication-style **600-dpi PNG + PDF** figures.
- `final_summary/` — final tables, model card, figure/output manifest and reproduction QA.

The downstream Qwen notebook reads only saved files from this directory. It does not retrain or alter Random Forest.
